## データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 電場のデータを弄ってみる

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr
import pandas as pd

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']
pt.timespan('2022-09-01/22:30:00', 1, keyword='hour')

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/2230-2330'

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi')
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi')

In [ ]:
E64_data_Ex = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform']
E64_data_Ey = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform']
B64_data    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi']

time_range_T = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E64_data_Ex = E64_data_Ex.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_Ey = E64_data_Ey.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data    = B64_data.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

print(E64_data_Ex)
print(E64_data_Ey)
print(B64_data)

In [ ]:
import xarray as xr
import numpy as np

# --- 0. 準備：変数名は質問に合わせている -----------------------------
Ex = E64_data_Ex.sortby('time')           # 時系列を昇順に
Ey = E64_data_Ey.sortby('time')
B  = B64_data.sortby('time')              # 64 Hz 磁場ベクトル

Bx = B.isel(v_dim=0)         # dims = ('time',)
By = B.isel(v_dim=1)
Bz = B.isel(v_dim=2)

# --- 1. E を B のタイムスタンプへ線形補間 ----------------------------
Ex_i = Ex.interp(time=B.time, method='linear')
Ey_i = Ey.interp(time=B.time, method='linear')
#   • 範囲外はデフォルトで NaN。必要なら
#     .interp(..., kwargs={'fill_value': 'extrapolate'}) など

# --- 2. Ez を計算 ----------------------------------------------------
EPS = 1e-12          # ゼロ割り回避用の閾値 (単位: nT)
Ez = xr.where(np.abs(Bz) > EPS, -(Ex_i*Bx + Ey_i*By)/Bz, np.nan)
Ez.name = 'E64Hz_dsi_Ez_waveform'

# --- 3. 必要なら Dataset にまとめる --------------------------------
ds = xr.Dataset({
    'Ex_dsi': Ex_i,
    'Ey_dsi': Ey_i,
    'Ez_dsi': Ez,
    'Bx_dsi': ('time', Bx.data),
    'By_dsi': ('time', By.data),
    'Bz_dsi': ('time', Bz.data),
}, coords={'time': B.time})

ds = ds.dropna(dim='time', how='any', subset=['Ex_dsi', 'Ey_dsi', 'Bx_dsi', 'By_dsi', 'Bz_dsi'])

ds

In [ ]:
import pytplot as pt
import os
import matplotlib.pyplot as plt

# ---- xarray → tplot へ格納 ----------------------------
for var in ['Ex_dsi', 'Ey_dsi', 'Ez_dsi', 'Bx_dsi', 'By_dsi', 'Bz_dsi']:
    pt.store_data(
        var,
        data={'x': ds['time'].values, 'y': ds[var].values},
    )

pt.options(['Ex_dsi', 'Ey_dsi', 'Ez_dsi'], 'ysubtitle', '[mV/m]')
pt.options('Ex_dsi', 'ytitle', 'E_x (DSI)')
pt.options('Ey_dsi', 'ytitle', 'E_y (DSI)')
pt.options('Ez_dsi', 'ytitle', 'E_z (DSI)')
pt.options(['Bx_dsi', 'By_dsi', 'Bz_dsi'], 'ysubtitle', '[nT]')
pt.options('Bx_dsi', 'ytitle', 'B_x (DSI)')
pt.options('By_dsi', 'ytitle', 'B_y (DSI)')
pt.options('Bz_dsi', 'ytitle', 'B_z (DSI)')

# ---- プロット -----------------------------------------
# 電場 3 成分 + 磁場 3 成分を上下 2 パネルに分けて描画
vars_to_plot = ['Ex_dsi', 'Ey_dsi', 'Ez_dsi', 'Bx_dsi', 'By_dsi', 'Bz_dsi']
if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_dsi.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        vars_to_plot,
        display=True   # 省略可（デフォルト）
    )

In [ ]:
import xarray as xr
import numpy as np

def dsi_to_fac(ds: xr.Dataset, window_sec: float = 100.0) -> xr.Dataset:
    """DSI→FAC 変換 (64 Hz データを想定)
    Parameters
    ----------
    ds : xr.Dataset
        必須変数: Ex_dsi, Ey_dsi, Ez_dsi, Bx_dsi, By_dsi, Bz_dsi
    window_sec : float, optional
        FAC 基底を決める移動平均時間 [s]
    Returns
    -------
    ds_out : xr.Dataset
        DSI + FAC の両方を含む Dataset
    """
    # --- 0. 時系列を昇順にしておく -----------------------------------
    ds = ds.sortby('time')

    # --- 1. 平均磁場 <B> の計算 --------------------------------------
    dt = (ds.time[1] - ds.time[0]).astype('timedelta64[ns]').astype(float) * 1e-9
    win_pts = int(window_sec / dt)
    B_dsi = ds[['Bx_dsi', 'By_dsi', 'Bz_dsi']].to_array('comp')      # (comp,time)
    print(B_dsi)

    B_avg = B_dsi.rolling(time=win_pts, center=True).mean('time')          # (comp,time)
    B_avg = B_avg.transpose('time', 'comp').values                   # (N,3)

    # --- 2. FAC 基底ベクトル -----------------------------------------
    z0 = np.array([0.0, 0.0, 1.0])
    e2 = np.cross(z0, B_avg)                                              # ⟂B & ⟂z
    # z0 // B のとき e2≈0 → 代わりに x軸とクロスする等の処理を追加しても良い
    e1 = np.cross(e2, B_avg)
    e3 = B_avg

    def _normalize(v):
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    e1_hat = _normalize(e1)
    e2_hat = _normalize(e2)
    e3_hat = _normalize(e3)

    # --- 3. 回転行列 R (time,3,3) とベクトル変換 ----------------------
    R = np.stack([e1_hat, e2_hat, e3_hat], axis=2)                   # columns=ê_i

    # DSI→FAC: v_fac = R · v_dsi
    E_dsi = ds[['Ex_dsi', 'Ey_dsi', 'Ez_dsi']].to_array('comp') \
              .transpose('time', 'comp').values                      # (N,3)
    B_dsi_np = B_dsi.transpose('time', 'comp').values               # (N,3)

    E_fac = np.einsum('tji,tj->ti', R, E_dsi)                        # (N,3)
    B_fac = np.einsum('tji,tj->ti', R, B_dsi_np)                     # (N,3)

    # --- 4. Dataset へ格納 ------------------------------------------
    ds_out = ds.copy()
    ds_out['Ex_fac'], ds_out['Ey_fac'], ds_out['Ez_fac'] = [
        (('time'), E_fac[:, k]) for k in range(3)]
    ds_out['Bx_fac'], ds_out['By_fac'], ds_out['Bz_fac'] = [
        (('time'), B_fac[:, k]) for k in range(3)]

    # attrs など必要ならここで設定
    return ds_out, R, e3_hat


# ---------- 使い方 -------------------------------------------------------
ds_fac, Rotation_tensor, e3_hat = dsi_to_fac(ds)

print(ds_fac[['Ex_fac', 'Ey_fac', 'Ez_fac']])
print(ds_fac[['Bx_fac', 'By_fac', 'Bz_fac']])

print(Rotation_tensor[150000, :, :])
print(e3_hat[150000, :])


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


# --- 角度 θ(t) = arccos( ê₃ · ẑ_DSI ) -----------------------------
e3_z     = Rotation_tensor[:, 2, 2]                  # (N,) ← 第3列・z成分
angle_deg = np.degrees(np.arccos(e3_z))              # 0–180° に丸め
t         = ds_fac.time.values                       # or ds.time.values

# フォルダが存在するか
if os.path.isdir(path_base_save_plot):
    # ── 存在する → プロットを作って保存のみ、Notebook上には表示しない
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, angle_deg, lw=1)
    ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
    ax.set_xlabel('Time')
    ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
    ax.grid(True)
    plt.tight_layout()

    # ファイル名は適宜変更
    save_path = os.path.join(path_base_save_plot, 'rotation_angle.png')
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

else:
    # ── 存在しない → 通常どおり表示だけ
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, angle_deg, lw=1)
    ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
    ax.set_xlabel('Time')
    ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
    ax.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import pytplot as pt
import numpy as np

# --- 1. pytplot 変数への格納 はそのまま ----
epoch = ds_fac.time.values.astype('datetime64[ns]').astype('int64') * 1e-9

for comp in ['x', 'y', 'z']:
    pt.store_data(f'E_fac_{comp}', data={'x': epoch, 'y': ds_fac[f'E{comp}_fac'].values})
    pt.store_data(f'B_fac_{comp}', data={'x': epoch, 'y': ds_fac[f'B{comp}_fac'].values})

    pt.options(f'E_fac_{comp}', 'ytitle', f'E_{comp} (FAC)\n[mV/m]')
    pt.options(f'B_fac_{comp}', 'ytitle', f'B_{comp} (FAC)\n[nT]')

for comp in ['perp', 'para']:
    if comp == 'perp':
        ds_E = np.sqrt(ds_fac['Ex_fac'].values**2 + ds_fac['Ey_fac'].values**2)
        ds_B = np.sqrt(ds_fac['Bx_fac'].values**2 + ds_fac['By_fac'].values**2)
    else:  # 'para'
        ds_E = ds_fac['Ez_fac'].values
        ds_B = ds_fac['Bz_fac'].values

    pt.store_data(f'E_fac_{comp}', data={'x': epoch, 'y': ds_E})
    pt.store_data(f'B_fac_{comp}', data={'x': epoch, 'y': ds_B})

    pt.options(f'E_fac_{comp}', 'ytitle', f'E_{comp} (FAC)\n[mV/m]')
    pt.options(f'B_fac_{comp}', 'ytitle', f'B_{comp} (FAC)\n[nT]')

# ------------------------------------------------------------
# 2. フォルダ存在チェック付きで tplot 発行
# ------------------------------------------------------------
vars_to_plot = [
    'E_fac_x', 'E_fac_y', 'E_fac_z',
    'B_fac_x', 'B_fac_y', 'B_fac_z'
]

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに保存のみ
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上の自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_fac.png')
    )
    # 完全に表示を防ぐために閉じる
    import matplotlib.pyplot as plt
    plt.close(fig)

else:
    # フォルダがない → 通常の表示のみ
    pt.tplot(
        vars_to_plot,
        display=True   # デフォルトなので省略可
    )


In [ ]:
import os
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt


# ---------- 1. Butterworth ハイパス設定 -----------------------
fs     = 64.0              # サンプリング周波数 [Hz]
cutoff = 1/6               # 0.25 Hz (spin 周期 8 s より少し高め)
sos    = butter(N=4, Wn=cutoff / (fs/2),
                btype='high', output='sos')

def highpass_segmented(y, sos):
    """
    欠損(NaN) が混じる配列を NaN 区間で分割しつつ filtfilt する関数
    """
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    padlen = 3 * (sos.shape[0] - 1)
    for s in segs:
        if len(s) <= padlen:
            continue
        out[s] = sosfiltfilt(sos, y[s])
    return out

# ---------- 2. B_fac_* にフィルタを掛けて新変数を登録 ---------------
epoch = ds_fac.time.values.astype('datetime64[ns]').astype('int64') * 1e-9

for comp in ['x', 'y', 'z']:
    for EB in ['E', 'B']:
        var_in  = f'{EB}_fac_{comp}'
        da      = pt.data_quants[var_in]
        hp_data = highpass_segmented(da.data, sos)
        var_out = f'{var_in}_hp'
        pt.store_data(var_out,
                      data={'x': da.time.values, 'y': hp_data})
        pt.options(var_out, 'ytitle',        f'{EB}_{comp} (HP)\n[nT]' if EB=='B' else f'{EB}_{comp} (HP)\n[mV/m]')

# ---------- 3. プロット（フォルダある→保存のみ／ない→表示） ----
vars_hp = [
    'E_fac_x_hp', 'E_fac_y_hp', 'E_fac_z_hp',
    'B_fac_x_hp', 'B_fac_y_hp', 'B_fac_z_hp'
]

if os.path.isdir(path_base_save_plot):
    # 存在する → 表示せずに保存のみ
    fig, axes = pt.tplot(
        vars_hp,
        display=False,
        return_plot_objects=True,
        save_png=os.path.join(path_base_save_plot, 'EB_fields_fac_hp.png')
    )
    plt.close(fig)
else:
    # 存在しない → 通常表示
    pt.tplot(
        vars_hp,
        display=True
    )


In [ ]:
E_x_hp = pt.data_quants['E_fac_x_hp']
B_x_hp = pt.data_quants['B_fac_x_hp']

print(E_x_hp)
print(B_x_hp)

In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt

sys.path.append("..")
import module_handmade.tdwavelet as tw
importlib.reload(tw)

# 存在チェック用にフォルダを作らない派ならコメントアウトしてOK
# os.makedirs(path_base_save_plot, exist_ok=True)

# 1. 各成分に対して CWT 計算とオプション設定
for EB, unit in [('B', 'nT^2/Hz'), ('E', '(mV/m)^2/Hz')]:
    for comp in ['x', 'y', 'z']:
        var_in = f'{EB}_fac_{comp}_hp'
        if var_in not in pt.data_quants:
            print(f'skip (no {var_in})')
            continue

        tw.tdwavelet(
            var_in,
            dt=1/64,
            s0=1/64*4,
            suffix='_cwt',
            zrange=[1e-6, 1e2]
        )

        var_out = f'{var_in}_cwt'
        # 軸設定
        pt.options(var_out, 'ylog', 1)
        pt.options(var_out, 'ztitle', 'PSD')
        pt.options(var_out, 'zsubtitle', unit)
        pt.options(var_out, 'ytitle', f'{EB}_{comp} (FAC)')
        pt.options(var_out, 'ysubtitle', '[Hz]')
        pt.options(var_out, 'colormap', 'turbo')
        pt.options(var_out, 'zrange', [1e-6, 1e3])
        pt.options(var_out, 'yrange', [1/6, 1E2])

# 2. タイムスパンを変えつつプロット／保存
time_windows = [
    np.datetime64('2022-09-01T22:30') + np.timedelta64(5, 'm')*n
    for n in range(12)
]

for t0 in time_windows:
    start_str = str(t0)             # "2022-09-01T22:30:00" など
    # tplot のタイムスパン設定
    pt.timespan(start_str, 5, keyword='minute')
    print(f"plotting window: {start_str}")

    vars_cwt = [
        'E_fac_x_hp_cwt', 'E_fac_y_hp_cwt', 'E_fac_z_hp_cwt',
        'B_fac_x_hp_cwt', 'B_fac_y_hp_cwt', 'B_fac_z_hp_cwt'
    ]

    if os.path.isdir(path_base_save_plot):
        # フォルダがあれば保存のみ
        # ファイル名に時刻を埋め込む（コロンがあるとダメなので置換）
        fn_time = start_str.replace(':', '').replace('T', '_')
        save_png = os.path.join(path_base_save_plot,
                                f'EB_fields_fac_cwt_{fn_time}.png')

        fig, axes = pt.tplot(
            vars_cwt,
            display=False,
            return_plot_objects=True,
            save_png=save_png
        )
        plt.close(fig)

    else:
        # フォルダがなければ表示のみ
        pt.tplot(
            vars_cwt,
            display=True
        )


In [ ]:
print(pt.data_quants['E_fac_x_hp_cwt'])
print(pt.data_quants['E_fac_y_hp_cwt'])
print(pt.data_quants['E_fac_z_hp_cwt'])
print(pt.data_quants['B_fac_x_hp_cwt'])
print(pt.data_quants['B_fac_y_hp_cwt'])
print(pt.data_quants['B_fac_z_hp_cwt'])

In [ ]:
e = pt.data_quants['E_fac_x_hp_cwt'].values
b = pt.data_quants['B_fac_x_hp_cwt'].values
print(np.sum(~np.isfinite(e)), np.sum(~np.isfinite(b)))


## 各成分のNoise(Median)を抽出

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pytplot as pt


# --- 1. PSD 変数リストと時間範囲 -----------------------------
vars_psd = [
    'E_fac_x_hp_cwt', 'E_fac_y_hp_cwt', 'E_fac_z_hp_cwt',
    'B_fac_x_hp_cwt', 'B_fac_y_hp_cwt', 'B_fac_z_hp_cwt'
]
t0, t1 = '2022-09-01T21:30:00', '2022-09-01T22:00:00'

# --- 2. 各変数ごとに median を計算して Dataset にまとめる ----
noise_da_dict = {}
for v in vars_psd:
    if v not in pt.data_quants:
        print(f'skip (no {v})')
        continue

    dq_cut = pt.data_quants[v].sel(time=slice(t0, t1))
    freq   = dq_cut.spec_bins.values
    med    = np.nanmedian(dq_cut.data, axis=0)

    med_da = xr.DataArray(
        data   = med,
        dims   = ['frequency'],
        coords = {'frequency': freq},
        name   = v
    )
    noise_da_dict[v] = med_da

noise_ds = xr.Dataset(noise_da_dict)
print(noise_ds)

# --- 3. プロット or 保存 --------------------------------------
fig, ax = plt.subplots(figsize=(6,4))
for key in noise_ds.data_vars:
    label = key.split('_')[0] + '_' + key.split('_')[2]
    ax.loglog(noise_ds['frequency'], noise_ds[key], label=label)

ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Median PSD')
ax.set_title(r'Noise floor 21:30-22:00 (median) (E: (mV/m)$^2$/Hz, B: (nT)$^2$/Hz)')
ax.grid(True, which='both', ls=':')
ax.legend()
ax.set_xlim(0.1, 100)
ax.set_ylim(1E-10, 2E1)
plt.tight_layout()

if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → 保存のみ
    # ファイル名にコロンが入るとまずいので除去
    fn = f"noise_median_{t0.replace(':','')}_{t1.replace(':','')}.png"
    save_path = os.path.join(path_base_save_plot, fn)
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"saved to {save_path}")
else:
    # フォルダが存在しない → 通常表示
    plt.show()



In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pytplot as pt

# ------------------------------------------------------------
# 2. clean_vars を pytplot に登録
# ------------------------------------------------------------
vars_psd    = [
    'E_fac_x_hp_cwt', 'E_fac_y_hp_cwt', 'E_fac_z_hp_cwt',
    'B_fac_x_hp_cwt', 'B_fac_y_hp_cwt', 'B_fac_z_hp_cwt'
]
frequency   = noise_ds['frequency'].values  # shape=(Nf,)
clean_vars  = []

for v in vars_psd:
    if v not in pt.data_quants:
        print(f'skip (no {v})')
        continue

    # 元の PSD (time x freq)
    dq        = pt.data_quants[v]
    noise_vec = noise_ds[v].values           # shape=(Nf,)
    cleaned   = dq.data - noise_vec[None, :]
    cleaned[cleaned <= 0] = np.nan           # log 表示できない小数値は NaN に

    # 新変数名
    v_out = v.replace('_cwt', '_clean')
    pt.store_data(
        v_out,
        data={'x': dq.time.values, 'y': cleaned, 'v': frequency}
    )

    # 軸タイトル・凡例・スケール設定
    # 例：ytitle_list に対応する順序でラベルを設定
    ytitle_list     = [
        'E_x (FAC)', 'E_y (FAC)', 'E_z (FAC)',
        'B_x (FAC)', 'B_y (FAC)', 'B_z (FAC)'
    ]
    zsubtitle_list  = [
        '(mV/m)^2/Hz', '(mV/m)^2/Hz', '(mV/m)^2/Hz',
        '(nT)^2/Hz',    '(nT)^2/Hz',    '(nT)^2/Hz'
    ]
    idx = vars_psd.index(v)

    pt.options(v_out, 'ytitle',      ytitle_list[idx])
    pt.options(v_out, 'ysubtitle',   '[Hz]')
    pt.options(v_out, 'ztitle',      'PSD (S–N)')
    pt.options(v_out, 'zsubtitle',   zsubtitle_list[idx])
    pt.options(v_out, 'colormap',    'turbo')
    pt.options(v_out, 'ylog',        1)        # 周波数軸 log
    pt.options(v_out, 'spec',        1)        # スペクトログラムモード
    pt.options(v_out, 'zlog',        1)        # カラー軸 log
    pt.options(v_out, 'zrange',     [1e-6, 1e3])
    pt.options(v_out, 'yrange',     [1/6, 1E2])

    clean_vars.append(v_out)

# ------------------------------------------------------------
# 3. タイムウィンドウを変えつつ保存 or 表示
# ------------------------------------------------------------
time_windows = [
    np.datetime64('2022-09-01T22:30') + np.timedelta64(5, 'm') * n
    for n in range(12)
]

for t0 in time_windows:
    ts = str(t0)  # '2022-09-01T22:30:00'
    pt.timespan(ts, 5, keyword='minute')
    print(f"Window: {ts}")

    if os.path.isdir(path_base_save_plot):
        # フォルダがある → 保存のみ
        fn_time = ts.replace(':', '').replace('T', '_')
        save_png = os.path.join(
            path_base_save_plot,
            f'EB_fields_fac_cwt_clean_{fn_time}.png'
        )

        # display=False で plt.show() を抑制、return_plot_objects=True で fig, axes を取得
        fig, axes = pt.tplot(
            clean_vars,
            display=False,             # tplot の出力そのままキャプチャ
            return_plot_objects=True,
            save_png=save_png
        )
        # 図を閉じて Notebook 上への自動表示も抑制
        plt.close(fig)
        print(f"Saved: {save_png}")

    else:
        # フォルダがない → Notebook 上に表示のみ
        pt.tplot(
            clean_vars,
            display=True
        )


import pyspedas as psp
import pytplot as pt

psp.erg.pwe_hfa(trange=time_range, level='l3')

import pytplot as pt
pt.tplot(['erg_pwe_hfa_l3_1min_ne_mgf'])

import pytplot as pt
number_density_l3 = pt.data_quants['erg_pwe_hfa_l3_1min_ne_mgf']
Bx_time = pt.data_quants['B_fac_x_hp_dpwrspc'].time
number_density_l3_i = number_density_l3.interp(time=Bx_time, method='linear')
pt.store_data('erg_pwe_hfa_l3_1min_ne_mgf_interp', data={'x': Bx_time.values, 'y': number_density_l3_i}, attr_dict=number_density_l3.attrs)
pt.tplot(['erg_pwe_hfa_l3_1min_ne_mgf_interp'])

In [ ]:
import os
import numpy as np
import pytplot as pt
import xarray as xr
import matplotlib.pyplot as plt

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/2230-2330'

# --- 1. B_total を計算し、ローリング平均用の win_pts を定義 ---
Bx = pt.data_quants['B_fac_x'].sel(time=slice(*time_range_T))
By = pt.data_quants['B_fac_y'].sel(time=slice(*time_range_T))
Bz = pt.data_quants['B_fac_z'].sel(time=slice(*time_range_T))

# numpy 配列ではなく xarray.DataArray のまま計算
B_total = np.sqrt(Bx.data**2 + By.data**2 + Bz.data**2) * 1e-9  # nT→T

# 各サンプル間隔を秒で取得
dt = (Bx.time.values[1] - Bx.time.values[0]).astype('timedelta64[ns]').astype(float) * 1e-9

# 例：100秒を移動平均したいなら
win_secs = 100.0
win_pts = int(win_secs / dt)

# xarray.DataArray に戻してローリング平均
B_total_da = xr.DataArray(
    data=B_total,
    coords={'time': Bx.time.values},
    dims=['time']
)
B_total_roll = B_total_da.rolling(time=win_pts, center=True).mean()

# --- 2. pytplot 変数に登録 ----------------------------------
pt.store_data(
    'B_total_T',
    data={
        'x': B_total_roll.time.values,
        'y': B_total_roll.values
    }
)
pt.options('B_total_T', 'ytitle', 'B (total)')
pt.options('B_total_T', 'ysubtitle', '[T]')

# --- 3. プロット or 保存 ----------------------------------
pt.timespan('2022-09-01/22:30:00', 1, keyword='hour')

if os.path.isdir(path_base_save_plot):
    save_path = os.path.join(path_base_save_plot, 'B_total_rolling.png')
    fig, axes = pt.tplot(
        'B_total_T',
        display=False,
        return_plot_objects=True,
        save_png=save_path
    )
    plt.close(fig)
    print(f"Saved plot to {save_path}")
else:
    pt.tplot('B_total_T', display=True)



density = number_density_l3_i.sel(time=slice(*time_range_T))
density_time = density.time.values

dt_ns = np.median(np.diff(density_time).astype('timedelta64[ns]').astype(float))
dt_sec = dt_ns * 1e-9          # 例: ≈ 4 s とか

B_total_avg = pt.data_quants['B_total_T-avg']
B_total_match = B_total_avg.interp(time=density_time)  # xarray の .interp

mu0 = 4*np.pi*1e-7
mp  = 1.6726219e-27
n_i = density.values * 1e6           # cm⁻³ → m⁻³

v_A = B_total_match.values / np.sqrt(mu0 * mp * n_i)

psp.avg_data('B_total_T', res=dt_sec)                 # → 'B_total_T-avg'
pt.store_data('Alfven_speed',
              data={'x': density_time, 'y': v_A})
pt.options('Alfven_speed', 'ytitle', 'Alfvén speed')
pt.options('Alfven_speed', 'ysubtitle', '[m/s]')
pt.tplot('Alfven_speed')

print(pt.data_quants['Alfven_speed'].time)

# 軌道データから、衛星速度(DSI)を導出

In [ ]:
import os
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/2230-2330'
# ──────────────────────────
# 1. 軌道＆磁場ロード＋GSE速度計算
# ──────────────────────────
psp.erg.orb(trange=time_range, level='l2', datatype='def')
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='gse')

pos   = pt.data_quants['erg_orb_l2_pos_gse'].data   # (Nt,3)[R_E]
t_pos = pt.data_quants['erg_orb_l2_pos_gse'].time.values

R_E_m = 6.378137e6
t_s   = t_pos.astype('datetime64[ns]').astype(float) * 1e-9
v_gse = np.gradient(pos * R_E_m, t_s, axis=0)

v_sc_gse = xr.DataArray(
    data=v_gse, dims=('time','v_dim'),
    coords={'time': t_pos},
    attrs={'units':'m/s','desc':'$V_{sc}$ (GSE)'}
)

pt.store_data('v_sc_gse', data={'x': t_s, 'y': v_gse})
pt.options('v_sc_gse', 'ytitle', r'$V_{sc}$ (GSE) [m/s]')
pt.options('v_sc_gse', 'legend_names', ['x','y','z'])

# ──────────────────────────
# 2. GSE→J2000→DSI 変換
# ──────────────────────────
# (1) GSE→J2000
psp.cotrans(
    name_in   = 'v_sc_gse',
    name_out  = 'v_sc_j2000',
    coord_in  = 'gse',
    coord_out = 'j2000'
)
pt.options('v_sc_j2000', 'ytitle', r'$V_{sc}$ (J2000) [m/s]')
pt.options('v_sc_j2000', 'legend_names', ['x','y','z'])

# (2) J2000→DSI
from pyspedas.projects.erg.satellite.erg.common.cotrans.dsi2j2000 import dsi2j2000

dsi2j2000(name_in='v_sc_j2000', name_out='v_sc_dsi', J20002DSI=True)
pt.options('v_sc_dsi', 'ytitle', r'$V_{sc}$ (DSI) [m/s]')
pt.options('v_sc_dsi', 'legend_names', ['x','y','z'])

# ──────────────────────────
# 3. 各プロットを「保存のみ／表示のみ」で切り替え
# ──────────────────────────
pt.timespan('2022-09-01/22:30:00', 1, keyword='hour')

for var in ['v_sc_gse','v_sc_j2000','v_sc_dsi']:
    if os.path.isdir(path_base_save_plot):
        png_path = os.path.join(path_base_save_plot, f'{var}.png')
        pt.tplot(var, save_png=png_path, display=False)
        plt.close('all')
        print(f"Saved: {png_path}")
    else:
        pt.tplot(var, display=True)


In [ ]:
V_sc_dsi = pt.data_quants['v_sc_dsi']

V_sc_dsi = V_sc_dsi.interp(time=pt.data_quants['B_fac_x_hp'].time, method='linear')

V_sc_fac_np = np.einsum('tji,tj->ti', Rotation_tensor, V_sc_dsi.values)  # (N,3) × (N,3,3) → (N,3)

V_sc_fac_perp_np = np.sqrt(V_sc_fac_np[:, 0]**2 + V_sc_fac_np[:, 1]**2)  # (N,)

pt.store_data('v_sc_fac',
              data={'x': V_sc_dsi.time.values, 'y': V_sc_fac_np},
              attr_dict=V_sc_dsi.attrs)

pt.store_data('v_sc_fac_perp',
              data={'x': V_sc_dsi.time.values, 'y': V_sc_fac_perp_np})

pt.options('v_sc_fac', 'char_size', 15)
pt.options('v_sc_fac', 'ytitle', r'$V_{\mathrm{sc}}$ (FAC)')
pt.options('v_sc_fac', 'ysubtitle', r'[m/s]')
pt.options('v_sc_fac', 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])

pt.options('v_sc_fac_perp', 'ytitle', r'$V_{\mathrm{sc}\perp}$ (FAC)')
pt.options('v_sc_fac_perp', 'char_size', 15)
pt.options('v_sc_fac_perp', 'ysubtitle', r'[m/s]')

vars_to_plot = ['v_sc_fac', 'v_sc_fac_perp']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'v_sc_fac_and_perp.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

## イオン流速($\approx$ MHD流速)、イオン温度→イオン熱速度、電子温度→ion acoustic speedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.erg.lepe(trange=time_range, datatype='3dflux', level='l2')
psp.erg.lepi(trange=time_range, datatype='3dflux', level='l2')
psp.erg.orb(trange=time_range, level='l2', datatype='def')

psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi'
)

pt.tplot_math.split_vec('erg_lepi_l2_3dflux_FPDU_velocity')

vars_mom = [
    'erg_lepe_l2_3dflux_FEDU_density',      # scalar (1/cm^3)
    'erg_lepi_l2_3dflux_FPDU_density',      # scalar (1/cm^3)
    'erg_lepi_l2_3dflux_FPDU_velocity_x',   # scalar (km/s)
    'erg_lepi_l2_3dflux_FPDU_velocity_y',   # scalar (km/s)
    'erg_lepi_l2_3dflux_FPDU_velocity_z',   # scalar (km/s)
    'erg_lepe_l2_3dflux_FEDU_avgtemp',      # scalar (eV)
    'erg_lepi_l2_3dflux_FPDU_avgtemp'       # scalar (eV)
]

for xyz in ['x', 'y', 'z']:
    v_out = f'erg_lepi_l2_3dflux_FPDU_velocity_{xyz}'
    pt.options(v_out, 'ytitle', r'$\mathrm{H}^{+}$ V'+f'{xyz}')
    pt.options(v_out, 'ysubtitle', '(DSI) [km/s]')

pt.options('erg_lepe_l2_3dflux_FEDU_density', 'ytitle', r'$\mathrm{e}^{-}$ Density')
pt.options('erg_lepi_l2_3dflux_FPDU_density', 'ytitle', r'$\mathrm{H}^{+}$ Density')
pt.options(['erg_lepe_l2_3dflux_FEDU_density', 'erg_lepi_l2_3dflux_FPDU_density'], 'ysubtitle', '[1/cm^3]')
pt.options('erg_lepe_l2_3dflux_FEDU_avgtemp', 'ytitle', r'$\mathrm{e}^{-}$ Temp.')
pt.options('erg_lepi_l2_3dflux_FPDU_avgtemp', 'ytitle', r'$\mathrm{H}^{+}$ Temp.')
pt.options(['erg_lepe_l2_3dflux_FEDU_avgtemp', 'erg_lepi_l2_3dflux_FPDU_avgtemp'], 'ysubtitle', '[eV]')
pt.options(['erg_lepe_l2_3dflux_FEDU_avgtemp', 'erg_lepi_l2_3dflux_FPDU_avgtemp'], 'yrange', [1E1, 1E4])
pt.options(['erg_lepe_l2_3dflux_FEDU_avgtemp', 'erg_lepi_l2_3dflux_FPDU_avgtemp'], 'ylog', True)

In [ ]:
pt.options('erg_lepi_l2_3dflux_FPDU_velocity', 'char_size', 15)
pt.options('erg_lepi_l2_3dflux_FPDU_velocity', 'ytitle',
           r'$V_{\mathrm{iflow}}$ (DSI, $\mathrm{H^{+}}$)')
pt.options('erg_lepi_l2_3dflux_FPDU_velocity', 'ysubtitle', r'[km/s]')
pt.options('erg_lepi_l2_3dflux_FPDU_velocity', 'legend_names',
           [r'$x$ (DSI)', r'$y$ (DSI)', r'$z$ (DSI)'])
pt.options('erg_lepi_l2_3dflux_FPDU_velocity', 'yrange', [-250, 250])


var = 'erg_lepi_l2_3dflux_FPDU_velocity'

if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → 保存のみ
    fn = os.path.join(path_base_save_plot, f'V_iflow_dsi.png')
    pt.tplot(
        var,
        display=False,    # Notebook 上の自動表示を抑制
        save_png=fn       # PNG をこのファイル名で保存
    )
    plt.close('all')     # 完全に閉じて自動表示も防止
    print(f"Saved plot to {fn}")
else:
    # フォルダがなければ通常表示のみ
    pt.tplot(var, display=True)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

V_ion_dsi = pt.data_quants['erg_lepi_l2_3dflux_FPDU_velocity']

V_ion_dsi_interp = V_ion_dsi.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

V_ion_fac_np = np.einsum('tji,tj->ti', Rotation_tensor, V_ion_dsi_interp.values)  # (N,3) × (N,3,3) → (N,3)

V_ion_fac_np_perp = np.sqrt(V_ion_fac_np[:, 0]**2 + V_ion_fac_np[:, 1]**2)  # (N,)

pt.store_data('erg_lepi_l2_3dflux_FPDU_velocity_fac',
              data={'x': V_ion_dsi_interp.time.values, 'y': V_ion_fac_np},
              attr_dict=V_ion_dsi_interp.attrs)

pt.store_data('erg_lepi_l2_3dflux_FPDU_velocity_fac_perp',
              data={'x': V_ion_dsi_interp.time.values, 'y': V_ion_fac_np_perp})

pt.tplot_math.split_vec('erg_lepi_l2_3dflux_FPDU_velocity_fac')

pt.options('erg_lepi_l2_3dflux_FPDU_velocity_fac_x', 'ytitle', r'$\mathrm{H}^{+}$ $V_{\mathrm{iflow}x}$')
pt.options('erg_lepi_l2_3dflux_FPDU_velocity_fac_y', 'ytitle', r'$\mathrm{H}^{+}$ $V_{\mathrm{iflow}y}$')
pt.options('erg_lepi_l2_3dflux_FPDU_velocity_fac_z', 'ytitle', r'$\mathrm{H}^{+}$ $V_{\mathrm{iflow}z}$')
pt.options(['erg_lepi_l2_3dflux_FPDU_velocity_fac_x',
            'erg_lepi_l2_3dflux_FPDU_velocity_fac_y',
            'erg_lepi_l2_3dflux_FPDU_velocity_fac_z'],
           'ysubtitle', '(FAC) [km/s]')
pt.options(['erg_lepi_l2_3dflux_FPDU_velocity_fac_x',
            'erg_lepi_l2_3dflux_FPDU_velocity_fac_y',
            'erg_lepi_l2_3dflux_FPDU_velocity_fac_z'],
           'legend_name', None)

pt.options('erg_lepi_l2_3dflux_FPDU_velocity_fac_perp', 'ytitle', r'$\mathrm{H}^{+}$ $V_{\mathrm{iflow}\perp}$')
pt.options('erg_lepi_l2_3dflux_FPDU_velocity_fac_perp',
           'ysubtitle', '(FAC) [km/s]')
pt.options('erg_lepi_l2_3dflux_FPDU_velocity_fac_perp', 'char_size', 15)

vars_fac = [
    'erg_lepi_l2_3dflux_FPDU_velocity_fac_x',
    'erg_lepi_l2_3dflux_FPDU_velocity_fac_y',
    'erg_lepi_l2_3dflux_FPDU_velocity_fac_z',
    'erg_lepi_l2_3dflux_FPDU_velocity_fac_perp'
]

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'V_iflow_fac.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
V_sys_fac_np_perp = np.sqrt((V_ion_fac_np[:, 0]-V_sc_fac_np[:, 0]*1E-3)**2 + (V_ion_fac_np[:, 1]-V_sc_fac_np[:, 1]*1E-3)**2)

pt.store_data('v_sys_fac_perp',
              data={'x': V_sc_dsi.time.values, 'y': V_sys_fac_np_perp})

pt.store_data('v_sys_fac',
              data={'x': V_sc_dsi.time.values, 'y': V_ion_fac_np-V_sc_fac_np*1E-3},
              attr_dict=V_sc_dsi.attrs)

pt.options('v_sys_fac', 'char_size', 15)
pt.options('v_sys_fac', 'ytitle', r'$V_{\mathrm{sys}}$ (FAC)')
pt.options('v_sys_fac', 'ysubtitle', r'[km/s]')
pt.options('v_sys_fac', 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])

pt.options('v_sys_fac_perp', 'ytitle', r'$V_{\mathrm{sys}\perp}$ (FAC)')
pt.options('v_sys_fac_perp', 'char_size', 15)
pt.options('v_sys_fac_perp', 'ysubtitle', r'[km/s]')

vars_fac = ['v_sys_fac', 'v_sys_fac_perp']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'V_sys_fac.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

Vperp_rolling = pt.data_quants['v_sys_fac_perp']
dt_i = (Vperp_rolling.time[1] - Vperp_rolling.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
win_pts_i = int(100.0 / dt_i)
Vperp_rolling = Vperp_rolling.rolling(time=win_pts_i, center=True).mean('time')

pt.store_data('v_sys_fac_perp_rolling',
              data={'x': Vperp_rolling.time.values, 'y': Vperp_rolling},
              attr_dict=Vperp_rolling.attrs)

vars_fac = 'v_sys_fac_perp_rolling'

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'V_sys_fac_perp_rolling.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

n_electron = pt.data_quants['erg_lepe_l2_3dflux_FEDU_density']

pt.options('erg_lepe_l2_3dflux_FEDU_density', 'yrange', [3E-2, 7E-1])

vars_fac = 'erg_lepe_l2_3dflux_FEDU_density'

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'electron_density.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

dt_e = (n_electron.time[1] - n_electron.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
win_pts_e = int(100.0 / dt_e)
n_electron = n_electron.rolling(time=win_pts_e, center=True).mean('time')

pt.store_data('erg_lepe_l2_3dflux_FEDU_density_rolling',
              data={'x': n_electron.time.values, 'y': n_electron},
              attr_dict=n_electron.attrs)

pt.options('erg_lepe_l2_3dflux_FEDU_density_rolling', 'yrange', [3E-2, 7E-1])

vars_fac = 'erg_lepe_l2_3dflux_FEDU_density_rolling'

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'electron_density_rolling.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

Temp_electron = pt.data_quants['erg_lepe_l2_3dflux_FEDU_avgtemp']
dt_e = (Temp_electron.time[1] - Temp_electron.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
print(dt_e)
win_pts_e = int(100.0 / dt_e)
Temp_electron = Temp_electron.rolling(time=win_pts_e, center=True).mean('time')
Temp_electron_interp = Temp_electron.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

Temp_proton = pt.data_quants['erg_lepi_l2_3dflux_FPDU_avgtemp']
dt_i = (Temp_proton.time[1] - Temp_proton.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
print(dt_i)
win_pts_i = int(100.0 / dt_i)
print(win_pts_i)
Temp_proton = Temp_proton.rolling(time=win_pts_i, center=True).mean('time')
Temp_proton_interp = Temp_proton.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

proton_mass = 1.6726219e-27  # kg
elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

ion_cyclo_freq = pt.data_quants['B_total_T'].interp(time=Temp_proton_interp.time, method='linear') * elementary_charge / (2 * np.pi * proton_mass)  # Hz

V_th_proton = np.sqrt(2 * Temp_proton_interp.values * elementary_charge / proton_mass) * 1E-3  # km/s (ion thermal speed)
C_s_proton = np.sqrt(Temp_electron_interp.values * elementary_charge / proton_mass) * 1E-3  # km/s (ion acoustic speed)
v_A        = pt.data_quants['B_total_T'].interp(time=Temp_proton_interp.time, method='linear') / np.sqrt(n_electron.interp(time=Temp_proton_interp.time, method='linear') * 1E6 * proton_mass * mu0) * 1E-3 # km/s

pt.store_data('V_th_proton',
              data={'x': Temp_proton_interp.time.values, 'y': V_th_proton})
pt.store_data('C_s_proton',
              data={'x': Temp_proton_interp.time.values, 'y': C_s_proton})
pt.store_data('ion_cyclo_freq',
              data={'x': Temp_proton_interp.time.values, 'y': ion_cyclo_freq})
pt.store_data('Alfven_speed',
              data={'x': Temp_proton_interp.time.values, 'y': v_A})

pt.options('V_th_proton', 'ytitle', r'Vthi ($\mathrm{H}^{+}$)')
pt.options('C_s_proton', 'ytitle', r'Cs ($\mathrm{H}^{+}$)')
pt.options('Alfven_speed', 'ytitle', r'$v_{\mathrm{A}}$')
pt.options('ion_cyclo_freq', 'ytitle', r'$f_{\mathrm{ci}}$ ($\mathrm{H}^{+}$)')
pt.options(['V_th_proton', 'C_s_proton', 'Alfven_speed'], 'ysubtitle', '[km/s]')
pt.options('ion_cyclo_freq', 'ysubtitle', '[Hz]')

vars_fac = ['V_th_proton', 'C_s_proton', 'Alfven_speed', 'ion_cyclo_freq']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_1.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
import pytplot as pt
import xarray as xr

v_A_    = pt.data_quants['Alfven_speed'] * 1E3
V_th_proton = pt.data_quants['V_th_proton'] * 1E3
C_s_proton = pt.data_quants['C_s_proton'] * 1E3

V_th_proton_interp = V_th_proton.interp(time=v_A_.time.values).values
C_s_proton_interp = C_s_proton.interp(time=v_A_.time.values).values

beta_i = (V_th_proton_interp / v_A_.values)**2E0
tau = (V_th_proton_interp / C_s_proton_interp)**2E0 / 2E0

pt.store_data('beta_i', data={'x': v_A_.time.values, 'y': beta_i})
pt.store_data('tau', data={'x': v_A_.time.values, 'y': tau})

pt.options('beta_i', 'ytitle', r'$\beta_{\mathrm{i}}$')
pt.options('tau', 'ytitle', r'$\tau$')


vars_fac = ['beta_i', 'tau']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_2.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pytplot as pt
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

target_time_str = '2022-09-01T22:37:00'

Bx_psd  = pt.data_quants['B_fac_x_hp_clean']
By_psd  = pt.data_quants['B_fac_y_hp_clean']
Ex_psd  = pt.data_quants['E_fac_x_hp_clean']
Ey_psd  = pt.data_quants['E_fac_y_hp_clean']
v_A_    = pt.data_quants['Alfven_speed'] * 1E3

t0      = np.datetime64(target_time_str)
idx     = np.argmin(np.abs(Bx_psd.time.values - t0))      # 最近傍インデックス
freq    = Bx_psd.v.values                      # 周波数軸 [Hz]

cutoff_freq = 1/6 # 1/6 Hz 以下の成分は除去する
mask = Bx_psd['spec_bins'] >= cutoff_freq          # True/False の配列
Bx_psd = Bx_psd.isel(v_dim=mask)
By_psd = By_psd.isel(v_dim=mask)
Ex_psd = Ex_psd.isel(v_dim=mask)
Ey_psd = Ey_psd.isel(v_dim=mask)
freq = freq[mask]  # 周波数軸も更新

B_perp2 = Bx_psd.values + By_psd.values
E_perp2 = Ex_psd.values + Ey_psd.values

# 1. 解析対象スロット（idx）の PSD を取り出し
B_perp2_slice = (Bx_psd[idx, :].values + By_psd[idx, :].values) * 1E-18         # [T^2/Hz]
E_perp2_slice = (Ex_psd[idx, :].values + Ey_psd[idx, :].values)                 # [(mV/m)^2/Hz]

# 2. 同時刻の Alfven 速度を取得（cm-3 → m-3 換算済みのものを使っている前提）
v_A_val = v_A_.interp(time=t0).values      # [m/s]


# 3. v_A^2 * E_perp^2  を計算
#    E の単位が mV/m なら V/m へ換算する必要あり → (1e-3)^2 = 1e-6
VA2_Bperp2_slice = (v_A_val**2) * (B_perp2_slice) * 1E6     # [(mV/m)^2/Hz]

print(VA2_Bperp2_slice.shape, E_perp2_slice.shape, freq.shape)

ratio_dimless = np.sqrt(E_perp2_slice * 1e-6 / B_perp2_slice) / v_A_val
#   E_perp2: (mV/m)2 → (V/m)2 に 1e-6 変換

# dispersion relation
ion_cyclo_freq = pt.data_quants['ion_cyclo_freq'].interp(time=t0).values  # [Hz]
V_ion_fac_perp = pt.data_quants['erg_lepi_l2_3dflux_FPDU_velocity_fac_perp'].interp(time=t0).values
V_th_proton = pt.data_quants['V_th_proton'].interp(time=t0).values
C_s_proton = pt.data_quants['C_s_proton'].interp(time=t0).values
E_B_v_A_ratio_VB = (1E0 + (freq / ion_cyclo_freq * V_th_proton / V_ion_fac_perp)**2E0) / np.sqrt(1E0 + (freq / ion_cyclo_freq)**2E0 * ((C_s_proton / V_ion_fac_perp)**2E0 + (V_th_proton / V_ion_fac_perp)**2E0))

tau = (V_th_proton / C_s_proton)**2E0 / 2E0
beta_i = (V_th_proton / v_A_val)**2E0

def dispersion_relation_ERMHD(freq):
    """
    すべて ndarray (もしくはブロードキャスト可能な形) を想定。
    戻り値は freq と同じ形の ndarray。
    """
    kperp_rhoi = (freq / ion_cyclo_freq) * (V_th_proton / V_ion_fac_perp)

    # 高k域の式
    high_val = kperp_rhoi * tau / np.sqrt((beta_i * (1.0 + tau) + 2.0 * tau) * (1.0 + tau))

    # 条件ごとの値を np.select で選ぶ
    conds    = [kperp_rhoi < np.sqrt(0.1),
                kperp_rhoi > np.sqrt(10.0)]
    choices  = [np.ones_like(kperp_rhoi), high_val]
    result   = np.select(conds, choices, default=np.nan)

    return result

E_B_v_A_ratio_ERMHD = dispersion_relation_ERMHD(freq)



# 4. プロット 2 枚（上下サブプロット） ------------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True,
                               gridspec_kw={'height_ratios': [1, 1]})

# --- (a) 既存の v_A2B⊥2 と E⊥2 -----------------------------------
ax1.loglog(freq, VA2_Bperp2_slice, label=r'$v_\mathrm{A}^{2}B_\perp^{2}$ PSD', color='b')
ax1.loglog(freq, E_perp2_slice,    label=r'$E_\perp^{2}$ PSD', color='orange')
ax1.set_ylabel(r'PSD [$(\mathrm{mV/m})^{2}$/Hz]')
ax1.set_title(f'PSD @ {np.datetime_as_string(Bx_psd.time.values[idx], unit="ns")}')
ax1.grid(True, which='both', ls='--', lw=0.5)
ax1.set_ylim(1e-4, 5e3)
ax1.set_xlim(1/8, 4e1)
ax1.legend()

# --- (b) 無次元比 √(E⊥2/B⊥2)/v_A -----------------------------------
ax2.loglog(freq, ratio_dimless, color='k', label=r'$\sqrt{E_\perp^{2}/B_\perp^{2}}/v_{\mathrm{A}}$')
ax2.loglog(freq, E_B_v_A_ratio_VB, color='green', label=r'dispersion relation (V-M)')
ax2.loglog(freq, E_B_v_A_ratio_ERMHD, color='blue', label=r'dispersion relation (ERMHD)')
ax2.set_xlabel('Frequency [Hz]')
ax2.set_ylabel(r'$\sqrt{E_\perp^{2}/B_\perp^{2}}/v_{\mathrm{A}}$')
ax2.grid(True, which='both', ls='--', lw=0.5)
ax2.set_ylim(5E-1, 1E2)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import os
import matplotlib as mpl
import sys
sys.path.append("..")
import module_handmade.psd_plotter_ERG as pp
importlib.reload(pp)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/22-24_cleaned_CWT_1sec_avg'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（モジュール関数を呼び出す）
# ------------------------------------------------------------
data_dict = pp.load_and_prepare_data(cutoff_freq=1/6)

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 解析対象の時間を決め、1秒ごとにループ
# ------------------------------------------------------------

def process_and_save_plot(t_start, data_dict, out_dir):
    """
    指定された単一の時刻について、スペクトルをプロットし、画像を保存する関数。
    """
    # モジュール関数を呼び出してプロットを作成
    # psd_plotter を pp としてインポートしている前提
    fig = pp.plot_freq_spectrum(t_start, data_dict, interval_sec=1, n_samples_mc=200) # MCサンプル数を減らしておく
    
    # figがNoneでなければ（データがあってプロットが作成されれば）保存
    if fig is not None:
        try:
            # ファイル名に使いやすいように文字列に変換
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=200)
        finally:
            # 保存に失敗しても、メモリ解放のために必ずクローズする
            plt.close(fig)

time_range = ['2022-09-01T22:30:00', '2022-09-01T23:30:00']
t_min, t_max = pd.to_datetime(time_range)

# pandas.date_rangeで1秒ごとのタイムスタンプを生成
time_steps = pd.date_range(start=t_min, end=t_max, freq='1s')

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_plot)(t_start, data_dict, out_dir) for t_start in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
from natsort import natsorted
import imageio                # ← v3 なら imageio.v2 を付けなくても同じ

img_dir = '/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/22-24_cleaned_dprt_kperprhoi'
png_files = natsorted([os.path.join(img_dir, f)
                       for f in os.listdir(img_dir)
                       if f.lower().endswith('.png')])

frames = [imageio.imread(f) for f in png_files]
out_path = os.path.join(img_dir, 'E_B_ratio_ERG.mp4')

# codec は imageio-ffmpeg の同梱バイナリが自動で使われる
imageio.mimsave(out_path, frames, fps=5, codec='libx264')
print('saved to', out_path)

## 横軸を$k_{\perp} \rho_{\mathrm{i}} \approx \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \, \frac{v_{\mathrm{thi}}}{V_{\perp \mathrm{flow}}}$に設定

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib as mpl
import sys
import gc # ガベージコレクションをインポート
from joblib import Parallel, delayed # joblibをインポート

sys.path.append("..")
import module_handmade.psd_plotter_ERG as pp
import importlib

importlib.reload(pp) # モジュールを修正した場合、リロードする

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/22-24_cleaned_CWT_1sec_avg_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（ループ前に一度だけ実行）
# ------------------------------------------------------------
print("Loading and preparing data...")
data_dict = pp.load_and_prepare_data(cutoff_freq=1/6)

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 並列処理のためのラッパー関数を定義
# ------------------------------------------------------------
def process_and_save_k_plot(t_start, data_dict, out_dir, k_range, n_bins, fit_range, dpi=200):
    """
    指定された単一の時刻について、kスペクトルをプロットし、画像を保存する関数。
    """
    fig = pp.plot_k_spectrum(
        t_start, 
        data_dict, 
        interval_sec=1, 
        k_range=k_range, 
        n_bins=n_bins,
        fit_range=fit_range
    )
    
    if fig is not None:
        try:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S_k_spec.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=dpi)
        finally:
            plt.close(fig) # メモリ解放
            gc.collect()   # ガベージコレクション

# ------------------------------------------------------------
# 3. 解析対象の時間を決め、並列処理で一気に実行
# ------------------------------------------------------------
time_range = ['2022-09-01T22:30:00', '2022-09-01T23:30:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq='1s')

# プロットのパラメータ
k_range_to_use      = (1e-1, 1e3)
n_bins_to_use       = 30
fit_range_to_use    = (3, 300)

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_k_plot)(
        t_start, 
        data_dict, 
        out_dir, 
        k_range=k_range_to_use, 
        n_bins=n_bins_to_use,
        fit_range=fit_range_to_use

    ) for t_start in time_steps
)

print('Finished saving all plots!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib as mpl
import sys
import gc # ガベージコレクションをインポート
from joblib import Parallel, delayed # joblibをインポート

sys.path.append("..")
import module_handmade.psd_plotter_ERG as pp
import importlib

importlib.reload(pp) # モジュールを修正した場合、リロードする

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/22-24_cleaned_CWT_30sec_avg_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（ループ前に一度だけ実行）
# ------------------------------------------------------------
print("Loading and preparing data...")
data_dict = pp.load_and_prepare_data(cutoff_freq=1/6)

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 並列処理のためのラッパー関数を定義
# ------------------------------------------------------------
def process_and_save_k_plot(t_start, data_dict, out_dir, k_range, n_bins, fit_range, dpi=200):
    """
    指定された単一の時刻について、kスペクトルをプロットし、画像を保存する関数。
    """
    fig = pp.plot_k_spectrum(
        t_start, 
        data_dict, 
        interval_sec=1, 
        k_range=k_range, 
        n_bins=n_bins,
        fit_range=fit_range
    )
    
    if fig is not None:
        try:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S_k_spec.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=dpi)
        finally:
            plt.close(fig) # メモリ解放
            gc.collect()   # ガベージコレクション

# ------------------------------------------------------------
# 3. 解析対象の時間を決め、並列処理で一気に実行
# ------------------------------------------------------------
time_range = ['2022-09-01T22:30:00', '2022-09-01T23:30:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq='30s')

# プロットのパラメータ
k_range_to_use      = (1e-1, 1e3)
n_bins_to_use       = 30
fit_range_to_use    = (3, 300)

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_k_plot)(
        t_start, 
        data_dict, 
        out_dir, 
        k_range=k_range_to_use, 
        n_bins=n_bins_to_use,
        fit_range=fit_range_to_use

    ) for t_start in time_steps
)

print('Finished saving all plots!')